In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


# 🌙 EALLIS: Enhanced Adaptive Low-Light Instance Segmentation — Kaggle Training

**Model**: Mask R-CNN + AWD + SCB + DSL (`MaskRCNNNoiseInv` with `ResNetAdaDSmoothPrior`)  
**Training**: COCO train2017 (from Kaggle dataset) with synthetic noise (SynCOCO)  
**Evaluation**: EALLIS test set  

---
⚠️ **Requires**: GPU accelerator. Settings → Accelerator → GPU T4 x2 or P100.

## 0. Global Paths (Run This First — Always)

> ⚠️ **Run this cell before any other cell.** All other cells depend on these variables.

In [ ]:
import os, sys, glob, re

DRIVE_BASE = '/content/drive/MyDrive'
COLAB_WORKSPACE = '/content'
REPO_DIR = os.path.join(COLAB_WORKSPACE, 'EALLIS')
SAVE_DIR = os.path.join(DRIVE_BASE, 'EALLIS_Output')   # wheels + checkpoints saved here
os.makedirs(SAVE_DIR, exist_ok=True)

# ─── Resume from a previous checkpoint? ───────────────────────────────────────
# Upload your .pth file as a Kaggle dataset, then set this path.
# Leave as None on the very first run.
RESUME_CHECKPOINT = None
# RESUME_CHECKPOINT = '/kaggle/input/my-eallis-checkpoint/epoch_6.pth'

# ─── Subset settings ──────────────────────────────────────────────────────────
USE_SUBSET   = True    # Set False for full COCO
SUBSET_RATIO = 0.25   # 25%

# ─── EALLIS availability flag (auto-set in Cell 7, override here if needed) ───
# This controls whether mid-training evaluation uses EALLIS or COCO val.
# Leave as None — Cell 7 will detect it automatically.
EALLIS_AVAILABLE = None

print(f'REPO_DIR : {REPO_DIR}')
print(f'SAVE_DIR : {SAVE_DIR}')
print(f'RESUME   : {RESUME_CHECKPOINT}')


## 1. Environment Check

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA:    {torch.version.cuda}')
print(f'Python:  {sys.version}')
print(f'GPU:     {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — enable GPU!"}')

# Setup COCO dataset
# Extract to local Colab storage (/content) for fast I/O during training
coco_zip = '/content/drive/MyDrive/datasets/coco.zip'
COCO_ROOT = '/content/coco2017'

if not os.path.exists('/content/coco2017/train2017') and not os.path.exists('/content/train2017') and not os.path.exists('/content/coco/train2017'):
    if os.path.exists(coco_zip):
        print(f"Unzipping {coco_zip} to /content ...")
        os.system(f"unzip -q {coco_zip} -d /content")
    else:
        print(f"WARNING: coco.zip not found at {coco_zip}!")

# Auto-detect extraction path based on how the zip was created
if os.path.exists('/content/coco2017/train2017'):
    COCO_ROOT = '/content/coco2017'
elif os.path.exists('/content/coco/train2017'):
    COCO_ROOT = '/content/coco'
elif os.path.exists('/content/train2017'):
    COCO_ROOT = '/content'

if os.path.exists(COCO_ROOT) and os.path.exists(os.path.join(COCO_ROOT, 'train2017')):
    print(f'✅ COCO root: {COCO_ROOT}')
else:
    print(f'❌ COCO dataset not found! Check if coco.zip extracted properly.')


## 2. Install Dependencies

Builds mmcv-full from source and caches the wheel in the output directory.

> 💡 **Reuse tip**: If you already downloaded `mmcv_full-1.7.2*.whl` from a previous run,
> upload it as a Kaggle dataset input. This cell will detect and install it instantly (skip the ~20 min build).

In [ ]:
import torch

torch_ver = torch.__version__.split('+')[0]
cuda_ver  = torch.version.cuda.replace('.', '')

# ─── Check for a pre-downloaded wheel in Drive ─────────────────────────
input_wheels = (['/content/drive/MyDrive/datasets/mmcv_full-1.7.2-cp312-cp312-linux_x86_64.whl'] if os.path.exists('/content/drive/MyDrive/datasets/mmcv_full-1.7.2-cp312-cp312-linux_x86_64.whl') else [])

try:
    import mmcv
    print(f'✅ mmcv-full {mmcv.__version__} already installed!')
except ImportError:
    # 1. Try pre-downloaded wheel from Drive (fastest)
    if input_wheels:
        print(f'📦 Installing pre-downloaded wheel from input: {input_wheels[0]}')
        os.system(f'pip install -q {input_wheels[0]}')

    # 2. Try previously built wheel saved in output
    elif glob.glob(os.path.join(SAVE_DIR, 'mmcv_full-1.7.2*.whl')):
        cached = glob.glob(os.path.join(SAVE_DIR, 'mmcv_full-1.7.2*.whl'))[0]
        print(f'📦 Found cached wheel in output, installing: {cached}')
        os.system(f'pip install -q {cached}')

    # 3. Build from source (~15-25 min)
    else:
        print(f'⏳ Building mmcv-full from source (~15-25 min)...')
        print(f'   PyTorch {torch_ver}, CUDA {cuda_ver}')
        os.system(f'pip wheel mmcv-full==1.7.2 -w /tmp/mmcv_wheels '
                  f'-f https://download.openmmlab.com/mmcv/dist/cu{cuda_ver}/torch{torch_ver}/index.html '
                  f'2>&1 | tail -5')
        built = glob.glob('/tmp/mmcv_wheels/mmcv_full-1.7.2*.whl')
        if built:
            os.system(f'pip install -q {built[0]}')
            import shutil
            shutil.copy2(built[0], SAVE_DIR)
            print(f'💾 Wheel saved to {SAVE_DIR} — download it to reuse next time!')
        else:
            print('⚠️  Wheel build failed, trying direct pip install...')
            os.system(f'pip install mmcv-full==1.7.2 '
                      f'-f https://download.openmmlab.com/mmcv/dist/cu{cuda_ver}/torch{torch_ver}/index.html')

    import mmcv
    print(f'✅ mmcv-full {mmcv.__version__} installed!')

os.system('pip install -q pycocotools scikit-learn terminaltables pretrainedmodels')


## 3. Clone EALLIS Repository

In [ ]:
os.chdir(COLAB_WORKSPACE)

if not os.path.exists(REPO_DIR):
    os.system(f'git clone https://github.com/itzaqeel/EALLIS.git {REPO_DIR}')
else:
    os.chdir(REPO_DIR)
    os.system('git pull')

os.chdir(REPO_DIR)


## 4. Install mmdetection

In [ ]:
os.chdir(os.path.join(REPO_DIR, 'mmdetection'))
os.system('pip install -q -e . --no-deps')
os.chdir(REPO_DIR)

if os.path.join(REPO_DIR, 'mmdetection') not in sys.path:
    sys.path.insert(0, os.path.join(REPO_DIR, 'mmdetection'))

import mmdet


## 5. Apply All Compatibility Fixes

Python 3.10/3.12, NumPy 2.x, mmcv 1.7.2, custom module imports.

In [ ]:
os.chdir(REPO_DIR)

# --- Fix 1: Bump mmcv version cap ---
init_file = 'mmdetection/mmdet/__init__.py'
with open(init_file, 'r') as f:
    content = f.read()
content = re.sub(r"mmcv_maximum_version\s*=\s*'[^']*'", "mmcv_maximum_version = '3.0.0'", content)
with open(init_file, 'w') as f:
    f.write(content)
print('[Fix 1] mmcv version cap updated.')

# --- Fix 2: Deprecated/removed imports (regex-safe) -----------------------
# Uses re.sub with (?<!\w) and (?!\w) word-boundary guards so that
# 'import imp' does NOT match inside 'import import_modules_from_strings'.
deprecated_imports = {
    r'(?<![\w.])import imp(?!\w)':                         '# import imp  # removed in Python 3.12',
    r'(?<![\w.])from os import pread(?!\w)':               '# from os import pread',
    r'(?<![\w.])from tokenize import group(?!\w)':         '# from tokenize import group',
    r'from numpy\.core\.fromnumeric import size':          '# from numpy.core.fromnumeric import size',
    r'from numpy\.core\.numeric import outer':             '# from numpy.core.numeric import outer',
    r'from numpy\.lib\.npyio import load':                 '# from numpy.lib.npyio import load',
    r'from numpy\.lib\.arraypad import pad':               'from numpy import pad',
    r'from numpy\.lib\.type_check import common_type':     'from numpy import common_type',
    r'from torch\.functional import _index_tensor_with_indices_list': '# from torch.functional import _index_tensor_with_indices_list',
    r'from numpy\.testing\._private\.utils import print_assert_equal': '# from numpy.testing._private.utils import print_assert_equal',
}
fix2_files = (
    glob.glob('mmdetection/mmdet/**/*.py', recursive=True) +
    glob.glob('mmdetection/tools/**/*.py', recursive=True) +
    glob.glob('mmdetection_custom_part/**/*.py', recursive=True) +
    glob.glob('utils/**/*.py', recursive=True)
)
fix2_count = 0
for py_file in fix2_files:
    try:
        with open(py_file, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
    except Exception:
        continue
    new_content = content
    for pattern, replacement in deprecated_imports.items():
        new_content = re.sub(pattern, replacement, new_content)
    if new_content != content:
        with open(py_file, 'w', encoding='utf-8') as f:
            f.write(new_content)
        fix2_count += 1
print(f'[Fix 2] Fixed deprecated imports in {fix2_count} files.')

# --- Fix 2b: Repair tools/train.py if corrupted by previous runs ------------
train_py = 'mmdetection/tools/train.py'
if os.path.exists(train_py):
    with open(train_py, 'r', encoding='utf-8') as f:
        tp = f.read()
    # Detect the corruption pattern and repair it
    corrupted = '# import imp  # removed in Python 3.12ort_modules_from_strings'
    repaired  = 'import import_modules_from_strings'
    if corrupted in tp:
        tp = tp.replace(corrupted, repaired)
        with open(train_py, 'w', encoding='utf-8') as f:
            f.write(tp)
        print('[Fix 2b] tools/train.py corruption repaired.')
    else:
        print('[Fix 2b] tools/train.py is clean.')

# --- Fix 2c: Add --local-rank alias (newer torch passes hyphen, old argparse uses underscore) ---
with open(train_py, 'r', encoding='utf-8') as f:
    tp = f.read()
old_arg = "parser.add_argument('--local_rank', type=int, default=0)"
new_arg = "parser.add_argument('--local_rank', '--local-rank', type=int, default=0)"
if old_arg in tp:
    tp = tp.replace(old_arg, new_arg)
    with open(train_py, 'w', encoding='utf-8') as f:
        f.write(tp)
    print('[Fix 2c] tools/train.py --local-rank alias added.')
elif new_arg in tp:
    print('[Fix 2c] tools/train.py --local-rank alias already present.')
else:
    print('[Fix 2c] WARNING: --local_rank argument not found in expected format.')

# --- Fix 3: Relative imports in custom_part ---
import_fixes = [
    ('mmdetection_custom_part/mmdet/models/backbones/resnet.py', 'from ..utils', 'from mmdet.models.utils'),
    ('mmdetection_custom_part/mmdet/models/backbones/resnet.py', 'from .cbam', 'from mmdet.models.backbones.cbam'),
    ('mmdetection_custom_part/mmdet/models/backbones/resnext.py', 'from ..utils', 'from mmdet.models.utils'),
    ('mmdetection_custom_part/mmdet/models/backbones/swin.py', 'from ...utils', 'from mmdet.utils'),
    ('mmdetection_custom_part/mmdet/models/backbones/swin.py', 'from ..utils.ckpt_convert', 'from mmdet.models.utils.ckpt_convert'),
    ('mmdetection_custom_part/mmdet/models/backbones/swin.py', 'from ..utils.transformer', 'from mmdet.models.utils.transformer'),
    ('mmdetection_custom_part/mmdet/models/dense_heads/maskformer_head.py', 'from .anchor_free_head', 'from mmdet.models.dense_heads.anchor_free_head'),
    ('mmdetection_custom_part/mmdet/models/dense_heads/mask2former_head.py', 'from .anchor_free_head', 'from mmdet.models.dense_heads.anchor_free_head'),
    ('mmdetection_custom_part/mmdet/models/detectors/two_stage.py', 'from .base', 'from mmdet.models.detectors.base'),
    ('mmdetection_custom_part/mmdet/models/detectors/maskformer.py', 'from .single_stage', 'from mmdet.models.detectors.single_stage'),
    ('mmdetection_custom_part/mmdet/models/seg_heads/base_semantic_head.py', 'from ..utils import interpolate_as', 'from mmdet.models.utils import interpolate_as'),
    ('mmdetection_custom_part/mmdet/models/seg_heads/panoptic_fpn_head.py', 'from ..utils import ConvUpsample', 'from mmdet.models.utils import ConvUpsample'),
]
for filepath, old_imp, new_imp in import_fixes:
    if os.path.exists(filepath):
        with open(filepath, 'r') as f:
            content = f.read()
        if old_imp in content:
            content = content.replace(old_imp, new_imp)
            with open(filepath, 'w') as f:
                f.write(content)
print('[Fix 3] Relative imports fixed.')

# --- Fix 4: Detector auxiliary imports ---
for det_file in [
    'mmdetection_custom_part/mmdet/models/detectors/mask_rcnn.py',
    'mmdetection_custom_part/mmdet/models/detectors/faster_rcnn_noise_inv.py'
]:
    if os.path.exists(det_file):
        with open(det_file, 'r') as f:
            content = f.read()
        content = content.replace('from ..backbones.aux_modules', 'from mmdet.models.backbones.aux_modules')
        content = content.replace('from ..backbones.multiscale_discriminator', 'from mmdet.models.backbones.multiscale_discriminator')
        content = content.replace('from ..backbones.lsid', 'from mmdet.models.backbones.lsid')
        with open(det_file, 'w') as f:
            f.write(content)
print('[Fix 4] Detector auxiliary imports fixed.')

# --- Fix 5: Force register_module() ---
count = 0
for py_file in glob.glob('mmdetection_custom_part/**/*.py', recursive=True):
    try:
        with open(py_file, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
    except Exception:
        continue
    if '.register_module()' in content:
        with open(py_file, 'w', encoding='utf-8') as f:
            f.write(content.replace('.register_module()', '.register_module(force=True)'))
        count += 1
print(f'[Fix 5] Forced registration in {count} files.')

# --- Fix 6: Rewrite __init__.py files ---
with open('mmdetection_custom_part/mmdet/models/backbones/__init__.py', 'w') as f:
    f.write("""from .resnet import ResNet, ResNetV1d, ResNetAdaD, ResNetAdaDSmoothPrior
from .resnext import ResNeXt
from .swin import SwinTransformer, SwinTransformerAdaD
from .convnext import ConvNeXt, ConvNeXtAdaD
__all__ = ['ResNet', 'ResNetV1d', 'ResNetAdaD', 'ResNetAdaDSmoothPrior',
           'ResNeXt', 'SwinTransformer', 'SwinTransformerAdaD',
           'ConvNeXt', 'ConvNeXtAdaD']
""")
with open('mmdetection_custom_part/mmdet/models/dense_heads/__init__.py', 'w') as f:
    f.write("""from .maskformer_head import MaskFormerHead
from .mask2former_head import Mask2FormerHead
__all__ = ['MaskFormerHead', 'Mask2FormerHead']
""")
with open('mmdetection_custom_part/mmdet/models/detectors/__init__.py', 'w') as f:
    f.write("""from .two_stage import TwoStageDetector
from .faster_rcnn import FasterRCNN
from .faster_rcnn_noise_inv import FasterRCNNNoiseInv
from .mask_rcnn import MaskRCNN, MaskRCNNNoiseInv as MaskRCNNNoiseInvDet
from .maskformer import MaskFormer
from .mask2former import Mask2Former
__all__ = ['TwoStageDetector', 'FasterRCNN', 'FasterRCNNNoiseInv',
           'MaskRCNN', 'MaskRCNNNoiseInvDet', 'MaskFormer', 'Mask2Former']
""")
models_init = 'mmdetection_custom_part/mmdet/models/__init__.py'
if os.path.exists(models_init):
    with open(models_init, 'r') as f:
        content = f.read()
    content = content.replace('from .necks import *', '# from .necks import *')
    with open(models_init, 'w') as f:
        f.write(content)
print('[Fix 6] __init__.py files rewritten.')

# --- Fix 7: NoiseModel camera_params path ---
noise_file = 'mmdetection/mmdet/datasets/pipelines/noisemodel/dark_noising.py'
with open(noise_file, 'r') as f:
    content = f.read()
if '~/code/mmdetection' in content:
    if 'import os\n' not in content:
        content = content.replace('import os.path as osp', 'import os\nimport os.path as osp')
    content = content.replace(
        """        if param_dir is None:
            try:
                self.param_dir = '~/code/mmdetection/mmdet/datasets/pipelines/noisemodel/camera_params'
            except:
                print('please specify the location of camera parameters, e.g., ~/code/mmdetection/mmdet/datasets/pipelines/noisemodel/camera_params')
                raise Exception""",
        """        if param_dir is None:
            self.param_dir = os.path.join(os.path.dirname(__file__), 'camera_params')""")
    with open(noise_file, 'w') as f:
        f.write(content)
    print('[Fix 7] NoiseModel camera_params path fixed.')
else:
    print('[Fix 7] NoiseModel path already fixed.')

# --- Fix 8: NumPy 2.x _sample_params inhomogeneous array ---
with open(noise_file, 'r') as f:
    content = f.read()
old_return = 'return np.array([K, color_bias, g_scale, G_scale, G_shape, R_scale, Q_step, saturation_level, ratio])'
new_return = 'return (K, color_bias, g_scale, G_scale, G_shape, R_scale, Q_step, saturation_level, ratio)'
if old_return in content:
    content = content.replace(old_return, new_return)
    with open(noise_file, 'w') as f:
        f.write(content)
    print('[Fix 8] NumPy 2.x _sample_params array fix applied.')
else:
    print('[Fix 8] _sample_params already fixed.')

print('\n✅ All fixes applied!')


## 6. Verify Custom Modules

In [ ]:
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
if os.path.join(REPO_DIR, 'mmdetection') not in sys.path:
    sys.path.insert(0, os.path.join(REPO_DIR, 'mmdetection'))

try:
    import mmdetection_custom_part.mmdet.models.detectors
    import mmdetection_custom_part.mmdet.datasets.pipelines.edge_target
    import mmdetection_custom_part.mmdet.models.backbones
    import mmdetection_custom_part.mmdet.models.plugins
    import mmdetection_custom_part.mmdet.models.seg_heads
    import mmdetection_custom_part.mmdet.models.losses
    print('✅ All custom modules imported successfully!')
except Exception as e:
    import traceback
    traceback.print_exc()


## 7. Setup Datasets

Symlinks Kaggle's pre-loaded COCO dataset into the expected `data/coco/` structure. **No copying needed — instant.**

> ℹ️ **EALLIS dataset**: If you haven't added the EALLIS dataset as Kaggle input, training will still work.
> Mid-training validation will automatically fall back to **COCO val2017** instead.
> You can evaluate on EALLIS separately in Cell 10 once the images are available.

In [ ]:
import shutil
os.chdir(REPO_DIR)

# --- COCO: Symlink from Kaggle input (read-only, no copy needed) ---
os.makedirs('data/coco', exist_ok=True)

for folder in ['annotations', 'train2017', 'val2017', 'test2017']:
    src = os.path.join(COCO_ROOT, folder)
    dst = os.path.join(REPO_DIR, 'data', 'coco', folder)
    if os.path.islink(dst):
        os.unlink(dst)
    elif os.path.isdir(dst):
        shutil.rmtree(dst)
    if os.path.exists(src):
        os.symlink(src, dst)
        print(f'  🔗 {folder}/ → {src}')
    else:
        print(f'  ⚠️  {src} not found')

train_count = len(os.listdir('data/coco/train2017'))
ann_files   = os.listdir('data/coco/annotations')
print(f'\n✅ COCO ready: {train_count:,} train images, {len(ann_files)} annotation files')

# --- EALLIS dataset ---
os.makedirs('data/eallis/annotations', exist_ok=True)
os.makedirs('data/eallis/images', exist_ok=True)

eallis_kaggle = '/content/drive/MyDrive/datasets/eallis'
if not os.path.exists(eallis_kaggle):
    eallis_kaggle = '/content/drive/MyDrive/datasets/eallis'

if os.path.exists(eallis_kaggle):
    for sub in ['annotations', 'images', 'JPEGImages']:
        src = os.path.join(eallis_kaggle, sub)
        if os.path.exists(src):
            target = 'data/eallis/images' if sub == 'JPEGImages' else f'data/eallis/{sub}'
            os.system(f'cp -rn {src}/* {target}/')
    eallis_img_count = len(os.listdir('data/eallis/images'))
    print(f'EALLIS images: {eallis_img_count} files')
else:
    print(f'ℹ️  EALLIS dataset not found at {eallis_kaggle}.')
    print(f'   Training will proceed — mid-training validation will use COCO val2017 instead.')
    eallis_img_count = 0

# JPEGImages symlink for EALLIS
jpeg_link = 'data/eallis/JPEGImages'
if os.path.islink(jpeg_link):
    os.unlink(jpeg_link)
if not os.path.exists(jpeg_link):
    os.symlink(os.path.abspath('data/eallis/images'), jpeg_link)
    print('🔗 data/eallis/JPEGImages → data/eallis/images')

# ─── Auto-detect EALLIS availability ──────────────────────────────────────────
# Check that both images AND the annotation file actually exist
eallis_ann = 'data/eallis/annotations/eallis_coco_JPG_test+1.json'
if eallis_img_count > 0 and os.path.exists(eallis_ann):
    EALLIS_AVAILABLE = True
    print('\n✅ EALLIS dataset ready — will evaluate on EALLIS after each epoch.')
else:
    EALLIS_AVAILABLE = False
    print('\n⚠️  EALLIS images/annotations not found.')


### 7b. Create 10% COCO Subset (Optional)

Set `USE_SUBSET = False` (in Cell 0) for full COCO training.

In [ ]:
import json, random
os.chdir(REPO_DIR)

if USE_SUBSET:
    ann_path    = 'data/coco/annotations/instances_train2017.json'
    os.makedirs('data/coco_subset', exist_ok=True)
    subset_path = 'data/coco_subset/instances_train2017_subset.json'

    print('Loading full COCO annotations...')
    with open(ann_path, 'r') as f:
        coco_data = json.load(f)

    all_images  = coco_data['images']
    num_subset  = int(len(all_images) * SUBSET_RATIO)
    random.seed(42)
    subset_images      = random.sample(all_images, num_subset)
    subset_img_ids     = set(img['id'] for img in subset_images)
    subset_annotations = [a for a in coco_data['annotations'] if a['image_id'] in subset_img_ids]

    subset_data = {
        'info':        coco_data.get('info', {}),
        'licenses':    coco_data.get('licenses', []),
        'images':      subset_images,
        'annotations': subset_annotations,
        'categories':  coco_data['categories']
    }
    with open(subset_path, 'w') as f:
        json.dump(subset_data, f)

    print(f'✅ {SUBSET_RATIO*100:.0f}% subset: {len(subset_images):,} images, {len(subset_annotations):,} annotations')
    print(f'   Saved to: {subset_path}')
else:
    print('Skipping subset generation (USE_SUBSET = False)')


## 8. Prepare Training Config

> 🔔 **AddNoisyImg (synthetic noise) is kept active.** The `AddNoisyImg` transform in the pipeline
> synthesises dark/noisy images on-the-fly using the PGRU noise model on CanonEOS5D4 camera params.
> This is the core of the DSL (Dark-Scene Learning) training strategy — do **not** remove it.

> ℹ️ If EALLIS is not available, the validation dataset in the config is automatically switched to **COCO val2017**
> so that training does not crash at the end of each epoch.

In [ ]:
import os
import sys
import glob
import re

os.chdir(REPO_DIR)

CONFIG_FILE = 'Configs/mask_rcnn_r50_fpn_caffe_AWD_SCB_DSL_SynCOCO2EALLIS.py'

with open(CONFIG_FILE, 'r') as f:
    config_content = f.read()

# Auto-detect GPU and optimize Batch Size & Workers
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ""

if 'A100' in gpu_name or 'V100' in gpu_name:
    # A100/V100 has huge VRAM, can safely handle BATCHSIZE = 16
    config_content = config_content.replace('BATCHSIZE = 8', 'BATCHSIZE = 16')
    print(f'Detected {gpu_name} -> Setting BATCHSIZE = 16')
    config_content = re.sub(r'workers_per_gpu\s*=\s*\w+', 'workers_per_gpu=8', config_content)
    print('workers_per_gpu = 8')
else:
    # T4 has 16GB VRAM, must use BATCHSIZE = 4 without FP16
    config_content = config_content.replace('BATCHSIZE = 8', 'BATCHSIZE = 4')
    print(f'Detected {gpu_name} -> Setting BATCHSIZE = 4')
    config_content = re.sub(r'workers_per_gpu\s*=\s*\w+', 'workers_per_gpu=4', config_content)
    print('workers_per_gpu = 4')


# ─── Evaluate every 3 epochs (saves ~15% time) ───────────────────────────────
config_content = re.sub(
    r'evaluation\s*=\s*dict\(interval=1,',
    'evaluation = dict(interval=3,',
    config_content)
print('Eval interval: every 3 epochs')

# ─── Checkpoint every 3 epochs, keep last 2 ──────────────────────────────────
config_content = re.sub(
    r'checkpoint_config\s*=\s*dict\(interval=\d+,\s*max_keep_ckpts=\d+\)',
    'checkpoint_config = dict(interval=3, max_keep_ckpts=2)',
    config_content)
print('Checkpoint interval: every 3 epochs')

# ─── Resume vs fresh start ────────────────────────────────────────────────────
if RESUME_CHECKPOINT and os.path.exists(RESUME_CHECKPOINT):
    config_content = re.sub(r"load_from\s*=\s*'[^']*'", "load_from = None", config_content)
    config_content = config_content.replace('resume_from = None', f"resume_from = '{RESUME_CHECKPOINT}'")
    print(f'Resuming from: {RESUME_CHECKPOINT}')
else:
    config_content = re.sub(r"load_from\s*=\s*'[^']*'", "load_from = None", config_content)
    print('Fresh training run.')

# ─── Full COCO or subset ──────────────────────────────────────────────────────
if USE_SUBSET:
    config_content = config_content.replace(
        "ann_file='data/coco/annotations/instances_train2017.json'",
        "ann_file='data/coco_subset/instances_train2017_subset.json'")
    print(f'Dataset: {SUBSET_RATIO*100:.0f}% COCO subset')
else:
    print('Dataset: FULL COCO (118k images)')

# ─── Validation dataset ───────────────────────────────────────────────────────
if not EALLIS_AVAILABLE:
    print('Val: COCO val2017 (EALLIS not found)')
    coco_val_override = """
_coco_val_fallback = dict(
    classes=('bicycle', 'chair', 'dining table', 'bottle', 'motorcycle', 'car', 'tv', 'bus'),
    type='CocoDataset',
    ann_file='data/coco/annotations/instances_val2017.json',
    img_prefix='data/coco/val2017/',
    pipeline=test_pipeline)
"""
    config_content += coco_val_override
    config_content = re.sub(r'val\s*=\s*test_lod_coco', 'val=_coco_val_fallback', config_content)
    config_content = re.sub(r'test\s*=\s*test_lod_coco', 'test=_coco_val_fallback', config_content)
else:
    print('Val: EALLIS test set')

# ─── work_dir, gt_edges, custom_imports ──────────────────────────────────────
config_content = config_content.replace("work_dir = './work_dir'", f"work_dir = '{SAVE_DIR}'")
config_content = config_content.replace(
    "keys=['img', 'noisy_img', 'gt_bboxes', 'gt_labels', 'gt_masks']",
    "keys=['img', 'noisy_img', 'gt_bboxes', 'gt_labels', 'gt_masks', 'gt_edges']")
config_content = config_content.replace("custom_imports = dict(imports=['mmdetection_custom_part.mmdet.datasets.pipelines.edge_target'], allow_failed_imports=False)", "")
custom_import_line = "custom_imports = dict(imports=['mmdetection_custom_part.mmdet.datasets.pipelines.edge_target', 'mmdetection_custom_part.mmdet.models.backbones.resnet'], allow_failed_imports=False)"
if custom_import_line not in config_content:
    config_content += '\n' + custom_import_line + '\n'

TRAIN_CONFIG = 'Configs/train_kaggle.py'
with open(TRAIN_CONFIG, 'w') as f:
    f.write(config_content)

print(f'\n Config saved: {TRAIN_CONFIG}')
print(f'  Epochs  : 12 | Eval: every 3 | Ckpt: every 3')
print(f'  Output  : {SAVE_DIR}')


## Run Automated Ablation
This will sequentially train the baseline, illum, and full variants for 5 epochs each, and evaluate the Boundary IoU.

In [ ]:
import os
os.environ['PYTHONPATH'] = f"/content/EALLIS:/content/EALLIS/mmdetection:{os.environ.get('PYTHONPATH', '')}"
import pandas as pd
import re
import glob

variants = {
    'baseline': {'use_eallis': 'False', 'use_edge': 'False'},
    'illum':    {'use_eallis': 'True',  'use_edge': 'False'},
    'full':     {'use_eallis': 'True',  'use_edge': 'True'}
}

work_root = '/content/drive/MyDrive/Ablation'
load_from = '/content/drive/MyDrive/EALLIS_checkpoints/Checkpoint1.pth'
epochs = 5
lr_step = 3

results = {}

for name, flags in variants.items():
    work_dir = f'{work_root}/{name}'
    os.makedirs(work_dir, exist_ok=True)
    
    # 1. Train
    if not os.path.exists(f'{work_dir}/epoch_{epochs}.pth'):
        print(f'\n========== TRAINING {name.upper()} ==========')
        !python mmdetection/tools/train.py {TRAIN_CONFIG} \
            --work-dir {work_dir} \
            --seed 42 \
            --cfg-options \
            model.backbone.use_eallis={flags['use_eallis']} \
            model.backbone.use_edge={flags['use_edge']} \
            runner.max_epochs={epochs} \
            lr_config.step=[{lr_step}] \
            evaluation.interval=1 \
            checkpoint_config.interval={epochs} \
            load_from={load_from} \
            resume_from=None
    else:
        print(f'\n========== {name.upper()} ALREADY TRAINED ==========')

    # 2. Extract Validation Metrics
    logs = sorted(glob.glob(f'{work_dir}/*.log.json'))
    bbox_mAP, segm_mAP, segm75 = None, None, None
    if logs:
        for line in open(logs[-1]):
            if not line.strip(): continue
            r = json.loads(line.strip())
            if r.get('mode') == 'val':
                bbox_mAP = r.get('bbox_mAP')
                segm_mAP = r.get('segm_mAP')
                segm75   = r.get('segm_mAP_75')

    # 3. Evaluate Boundary IoU
    biou = None
    ckpts = sorted(glob.glob(f'{work_dir}/epoch_*.pth'))
    if ckpts:
        dumped = glob.glob(f'{work_dir}/*.py')
        cfg_to_use = dumped[0] if dumped else TRAIN_CONFIG
        print(f'Running Boundary IoU for {name}...')
        out = !python tools/eval_boundary.py --config {cfg_to_use} --checkpoint {ckpts[-1]}
        out_str = '\n'.join(out)
        m = re.search(r'mean Boundary IoU\s*:\s*([0-9.]+)', out_str)
        if m: biou = float(m.group(1))
        else: print('Boundary IoU extraction failed!', out_str[-500:])

    results[name] = {'bbox_mAP': bbox_mAP, 'segm_mAP': segm_mAP, 'segm_AP75': segm75, 'boundary_iou': biou}

# 4. Save and Print Results
df = pd.DataFrame.from_dict(results, orient='index')
df.index.name = 'variant'
csv_path = f'{work_root}/ablation_results.csv'
df.to_csv(csv_path)
print(f'\nSaved to {csv_path}')
from IPython.display import display
display(df)


## View Ablation Results

In [ ]:
import pandas as pd
